Week 15 · Day 4 — Retrieval + Generation (RAG Loop)
Why this matters

Today you’ll connect the dots: user query → retrieve docs → feed into an LLM → generate grounded answer. This is the RAG pipeline in action. Even with a small dataset, this shows how “chat-with-docs” apps really work.

Theory Essentials

Pipeline:

Embed query → vector.

Search index → top-k docs.

Build prompt with docs as context.

LLM generates an answer.

LLM role: synthesize info, not just retrieve.

Grounding: ensures answer is tied to retrieved docs.

Limitations: if retrieval fails, LLM may hallucinate.

In [20]:
# Setup
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

# Docs
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London.",
    "The Acropolis is in Athens."
]

# Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs).astype("float32")

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# Retrieval function
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)
    return [docs[i] for i in I[0]]

# Mini LLM (using Hugging Face pipeline with distil GPT-2 for demo)
generator = pipeline("text-generation", model="distilgpt2")

def rag_answer(query, k=1):
    context = retrieve(query, k)
    prompt = f"Answer the question using the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"
    output = generator(prompt, max_new_tokens=10, do_sample=True,  temperature=0.1)[0]['generated_text']
    return output

# Example
print(rag_answer("Where is the Colosseum located?"))


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer the question using the context.

Context: ['The Colosseum is in Rome.']

Question: Where is the Colosseum located?
Answer: The Colosseum is located in Rome.


1) Core (10–15 min)

Task: Ask “Where is Big Ben?” and see if the answer includes London.

In [24]:
print(rag_answer("Where is Big Ben located?"))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer the question using the context.

Context: ['Big Ben is in London.']

Question: Where is Big Ben located?
Answer: [Big Ben is in London.]
Question:


2) Practice (10–15 min)

Task: Modify rag_answer to also print the retrieved context docs before the answer.

In [25]:

print(rag_answer("Where is the Prado Museum?"))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer the question using the context.

Context: ['The Prado Museum is in Madrid.']

Question: Where is the Prado Museum?
Answer: The Prado Museum is in Madrid.
Question


3) Stretch (optional, 10–15 min)

Task: Add a new document (“The Louvre is in Paris.”). Then query “Which museums are in Paris?” and see if the model uses both docs.

In [26]:
docs.append("The Louvre is in Paris.")
embeddings = model.encode(docs).astype("float32")
index.add(embeddings[-1:].astype("float32"))

print(rag_answer("Which museums are in Paris?", k=3))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer the question using the context.

Context: ['The Louvre is in Paris.', 'The Eiffel Tower is in Paris.', 'The Prado Museum is in Madrid.']

Question: Which museums are in Paris?
Answer: The Louvre is in Paris.
Question:


Mini-Challenge (≤40 min)

Build Your First RAG Demo

Create a qa(query) function:

Retrieves top-3 docs.

Shows context.

Generates an answer with the LLM.

Run it on at least 3 different queries.

Acceptance Criteria:

Retrieved docs are printed clearly.

Generated answer makes sense and references correct context.

Works with newly added docs.

In [27]:
# Mini-Challenge — First RAG Demo (simple & reliable on CPU)

# 1) Setup
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Embeddings model
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

# Use inner product (cosine) -> normalize vectors first
def _normalize(x: np.ndarray) -> np.ndarray:
    x = x.astype("float32")
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norms

# 2) Build the tiny knowledge base + FAISS index
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London.",
    "The Acropolis is in Athens."
]

def build_index(docs_list):
    embs = emb_model.encode(docs_list)
    embs = _normalize(embs)
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)     # cosine via inner product on normalized vectors
    index.add(embs)
    return docs_list[:], embs, index

docs, doc_embs, index = build_index(docs)

# 3) Retrieval (top-k)
def retrieve(query: str, k: int = 3):
    q = emb_model.encode([query])
    q = _normalize(q)
    D, I = index.search(q, k)
    pairs = [(docs[i], float(D[0][j])) for j, i in enumerate(I[0])]
    return pairs  # [(doc_text, similarity_score), ...]

# 4) Tiny LLM (extractive QA is best here)
qa_pipe = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

def qa(query: str, k: int = 3):
    top = retrieve(query, k=k)
    context = " ".join([t for t, _ in top])
    # Show context clearly (acceptance criterion)
    print(f"\nQuestion: {query}\nContext (top-{k}):")
    for t, s in top:
        print(f" - {t}  (sim={s:.3f})")
    # Generate grounded answer
    ans = qa_pipe(question=query, context=context)["answer"]
    print("Answer:", ans)
    return ans

# 5) Try at least 3 queries
queries = [
    "Where is the Colosseum located?",
    "In which city is Big Ben?",
    "What city has the Prado Museum?"
]
for q in queries:
    qa(q, k=3)

# 6) Add new docs and show it still works
def add_docs(new_docs):
    global docs, doc_embs, index
    docs += list(new_docs)
    # simple rebuild (fine for this mini-challenge)
    docs, doc_embs, index = build_index(docs)

add_docs(["The Leaning Tower of Pisa is in Pisa."])

qa("Where is the Leaning Tower of Pisa?", k=3)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

c:\AI-Mastery\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Device set to use cpu



Question: Where is the Colosseum located?
Context (top-3):
 - The Colosseum is in Rome.  (sim=0.842)
 - The Acropolis is in Athens.  (sim=0.398)
 - The Prado Museum is in Madrid.  (sim=0.274)
Answer: Rome

Question: In which city is Big Ben?
Context (top-3):
 - Big Ben is in London.  (sim=0.857)
 - The Eiffel Tower is in Paris.  (sim=0.159)
 - The Prado Museum is in Madrid.  (sim=0.143)
Answer: London

Question: What city has the Prado Museum?
Context (top-3):
 - The Prado Museum is in Madrid.  (sim=0.850)
 - The Colosseum is in Rome.  (sim=0.274)
 - The Acropolis is in Athens.  (sim=0.254)
Answer: Madrid

Question: Where is the Leaning Tower of Pisa?
Context (top-3):
 - The Leaning Tower of Pisa is in Pisa.  (sim=0.888)
 - The Eiffel Tower is in Paris.  (sim=0.415)
 - The Acropolis is in Athens.  (sim=0.275)
Answer: Pisa


'Pisa'

Notes / Key Takeaways

RAG = retrieval + generation loop.

The retriever ensures factual grounding.

The generator provides natural language synthesis.

Prompt design matters (good context → better answers).

Small demo today = foundation for scalable RAG apps.

Reflection

How does RAG reduce hallucinations compared to a plain LLM?

What happens if retrieval surfaces irrelevant docs?

How does RAG reduce hallucinations compared to a plain LLM?

A plain LLM answers from its training memory, which may be outdated or wrong.

RAG grounds the answer in retrieved documents, so the model can quote or summarize actual text instead of inventing facts.

What happens if retrieval surfaces irrelevant docs?

The LLM will base its answer on the wrong context, leading to misleading or incorrect outputs.

In practice, this means retrieval quality is as important as generation quality — if the context is wrong, the final answer will be too.